Create the Spark session


In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("CampusPlacementPrediction").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

Load placement data into a Spark DataFrame


In [ ]:
file_path = "/content/campus_placement_new.csv"
data = spark.read.csv(file_path, header=True, inferSchema=True)

data.show(5)
data.printSchema()
print("Rows:", data.count(), "| Columns:", len(data.columns))

+----------+------+--------------------+----+--------------+---------------+-------------------+---------+
|Student_ID|Gender|      Specialization|CGPA|Aptitude_Score|Technical_Skill|Communication_Skill|Placement|
+----------+------+--------------------+----+--------------+---------------+-------------------+---------+
|     S0001|Female|    Computer Science|6.22|          58.0|           61.0|               80.0|       No|
|     S0002|  Male|Business Administ...| 7.5|          69.0|           54.0|               48.0|      Yes|
|     S0003|  Male|    Computer Science|7.52|          62.0|           59.0|               66.0|       No|
|     S0004|Female|   Civil Engineering|6.42|          50.0|           38.0|               69.0|       No|
|     S0005|Female|    Computer Science|6.95|          NULL|           62.0|               83.0|      Yes|
+----------+------+--------------------+----+--------------+---------------+-------------------+---------+
only showing top 5 rows
root
 |-- Stu

Handle null values in marks and skill-test columns

In [ ]:
from pyspark.sql.functions import col, sum as spark_sum, when
print("Null count per column (before cleaning):")
data.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in data.columns
]).show()

Null count per column (before cleaning):
+----------+------+--------------+----+--------------+---------------+-------------------+---------+
|Student_ID|Gender|Specialization|CGPA|Aptitude_Score|Technical_Skill|Communication_Skill|Placement|
+----------+------+--------------+----+--------------+---------------+-------------------+---------+
|         0|     0|             0|  58|            74|             63|                 60|        0|
+----------+------+--------------+----+--------------+---------------+-------------------+---------+



In [ ]:
from pyspark.ml.feature import Imputer

numeric_cols = ["CGPA", "Aptitude_Score", "Technical_Skill", "Communication_Skill"]

imputer = Imputer(
    inputCols=numeric_cols,
    outputCols=numeric_cols,
    strategy="median"
)
data = imputer.fit(data).transform(data)

print("Null count per column (after cleaning):")
data.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in data.columns
]).show()

Null count per column (after cleaning):
+----------+------+--------------+----+--------------+---------------+-------------------+---------+
|Student_ID|Gender|Specialization|CGPA|Aptitude_Score|Technical_Skill|Communication_Skill|Placement|
+----------+------+--------------+----+--------------+---------------+-------------------+---------+
|         0|     0|             0|   0|             0|              0|                  0|        0|
+----------+------+--------------+----+--------------+---------------+-------------------+---------+



In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

data = data.withColumn("label", when(col("Placement") == "Yes", 1.0).otherwise(0.0))

In [ ]:
gender_indexer = StringIndexer(inputCol="Gender", outputCol="GenderIndex")
spec_indexer = StringIndexer(inputCol="Specialization", outputCol="SpecIndex")

In [ ]:
encoder = OneHotEncoder(
    inputCols=["GenderIndex", "SpecIndex"],
    outputCols=["GenderVec", "SpecVec"]
)


In [ ]:
from pyspark.ml.feature import VectorAssembler

feature_cols = numeric_cols + ["GenderVec", "SpecVec"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

In [ ]:
train, test = data.randomSplit([0.7, 0.3], seed=42)
print("Training rows:", train.count(), "| Testing rows:", test.count())

Training rows: 1102 | Testing rows: 398


In [ ]:
total = train.count()
n_placed = train.filter(col("label") == 1.0).count()
n_not = total - n_placed
w_placed = total / (2.0 * n_placed)
w_not = total / (2.0 * n_not)
print(f"Class weights -> Placed: {w_placed:.3f} | Not Placed: {w_not:.3f}")

train = train.withColumn("classWeight",
                         when(col("label") == 1.0, w_placed).otherwise(w_not))

Class weights -> Placed: 0.870 | Not Placed: 1.175


In [ ]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",
    numTrees=100,
    maxDepth=5,
    seed=42
)

In [ ]:
pipeline = Pipeline(stages=[gender_indexer, spec_indexer, encoder, assembler, rf])

model = pipeline.fit(train)             # learn from training data
predictions = model.transform(test)     # predict on unseen test data

predictions.select("Student_ID", "label", "prediction", "probability").show(10, truncate=False)

+----------+-----+----------+----------------------------------------+
|Student_ID|label|prediction|probability                             |
+----------+-----+----------+----------------------------------------+
|S0003     |0.0  |1.0       |[0.4914255826192996,0.5085744173807004] |
|S0007     |1.0  |0.0       |[0.585595557131164,0.41440444286883604] |
|S0009     |0.0  |0.0       |[0.7146783427185542,0.28532165728144576]|
|S0010     |0.0  |0.0       |[0.9151761715816409,0.0848238284183591] |
|S0014     |1.0  |1.0       |[0.14338407100778006,0.8566159289922198]|
|S0015     |0.0  |0.0       |[0.7578415333623456,0.24215846663765445]|
|S0016     |1.0  |1.0       |[0.2691944920900016,0.7308055079099984] |
|S0020     |1.0  |1.0       |[0.32157167152514515,0.6784283284748549]|
|S0022     |0.0  |0.0       |[0.894236635621523,0.10576336437847705] |
|S0024     |0.0  |0.0       |[0.8879170731195002,0.11208292688049983]|
+----------+-----+----------+----------------------------------------+
only s

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

def metric(name):
    return MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName=name
    ).evaluate(predictions)

print("========== Overall (weighted across both classes) ==========")
print("Accuracy  :", round(metric("accuracy"), 4))
print("Precision :", round(metric("weightedPrecision"), 4))
print("Recall    :", round(metric("weightedRecall"), 4))
print("F1-score  :", round(metric("f1"), 4))

========== Overall (weighted across both classes) ==========
Accuracy  : 0.8116
Precision : 0.8122
Recall    : 0.8116
F1-score  : 0.8116


In [ ]:
print("\n========== Per class ==========")
for cls, name in [(1.0, "Placed (Yes)"), (0.0, "Not Placed (No)")]:
    p = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                          metricName="precisionByLabel", metricLabel=cls).evaluate(predictions)
    r = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                          metricName="recallByLabel", metricLabel=cls).evaluate(predictions)
    f = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                          metricName="fMeasureByLabel", metricLabel=cls).evaluate(predictions)
    print(f"{name:16s} precision={p:.4f}  recall={r:.4f}  f1={f:.4f}")


========== Per class ==========
Placed (Yes)     precision=0.8308  recall=0.8029  f1=0.8166
Not Placed (No)  precision=0.7919  recall=0.8211  f1=0.8062


In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")   # lets the script save the image without a display; remove in Jupyter
import matplotlib.pyplot as plt
import seaborn as sns

# Count (actual, predicted) pairs in Spark, then bring the tiny 2x2 result to Python
rows = predictions.groupBy("label", "prediction").count().collect()
cm = np.zeros((2, 2), dtype=int)
for r in rows:
    cm[int(r["label"]), int(r["prediction"])] = r["count"]

tn, fp, fn, tp = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
print("\nConfusion matrix  [rows = actual, columns = predicted]")
print(cm)
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Not Placed", "Placed"],
            yticklabels=["Not Placed", "Placed"],
            annot_kws={"size": 16})
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Campus Placement - Random Forest Confusion Matrix")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()


Confusion matrix  [rows = actual, columns = predicted]
[[156  34]
 [ 41 167]]
TN=156  FP=34  FN=41  TP=167


In [ ]:
rf_model = model.stages[-1]
spec_labels = model.stages[1].labels        # branch names in index order
gender_labels = model.stages[0].labels
# OneHotEncoder drops the last category by default, so it has (n-1) slots
names = (numeric_cols
         + [f"Gender_{g}" for g in gender_labels[:-1]]
         + [f"Spec_{s}" for s in spec_labels[:-1]])

importances = sorted(zip(names, rf_model.featureImportances.toArray()),
                     key=lambda x: -x[1])
print("\nFeature importance:")
for n, v in importances:
    print(f"{n:30s} {v:.4f}")

spark.stop()


Feature importance:
CGPA                           0.4091
Aptitude_Score                 0.2873
Technical_Skill                0.2014
Communication_Skill            0.0662
Spec_Information Technology    0.0079
Spec_Electronics               0.0076
Spec_Computer Science          0.0072
Spec_Business Administration   0.0062
Gender_Male                    0.0045
Spec_Mechanical                0.0026
